# rounds

> Capture a session from any host, review it, accept it, and start the next session from it.

In [ ]:
#| default_exp rounds

In [ ]:
#| export
from pathlib import Path
import json, re, shlex, subprocess, sys

from aidialog.dialog import Dialog, snote, sprompt
from aidialog.hist import chat2dlg, dlg2chat
from aidialog.ipynb import read_ipynb, write_ipynb
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult
from fastcore.script import call_parse
from llmsurgery import ant, oai
from llmsurgery.sess import path_dlg, sess_dlg
from urai import ToolCall, mk_tool_res_msg

from drona.core import RAMABANA_HISTORY, assess_history, read_history, report

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
import tempfile

A round is one reviewed conversation that later ones start from, held as an Aidialog notebook: a
host session becomes one, a person edits it in Leela, and it compiles back out to whichever host
runs next.

Its state sits under a `drona` metadata key. `capture` writes `review`, `accept` records a named
reviewer and writes `accepted`, and every compiler refuses anything less.

In [ ]:
#| export
REVIEW_KEY = 'drona'
HOSTS = ('ramabana', 'claude', 'codex')
ROUND_REVISION = 2
BOOTSTRAP_DETAIL = 600
PENDING = '<output result="pending" reason="incomplete"/>'
MEDIA = re.compile(r'<media\b[^>]*>.*?</media>|!\[\]\(attachment:[^)]*\)', re.S)
REVIEW_NOTE = ('# Drona review\n\n'
               'Edit this dialog in Leela. Delete the detours and anything private, keep the route a '
               'later model should imitate, then run `drona-accept` on it.')

def drona_version():
    "The package version and round revision stamped into every accepted round."
    from drona import __version__
    return f'{__version__}:{ROUND_REVISION}'

def _clip(text, n):
    s = str(text or '')
    return s if len(s) <= n else s[:n] + f'\n…[{len(s)-n} more chars]'

def _plain(text):
    "Text with the markers for attachments that no longer travel with it removed."
    return MEDIA.sub('', str(text or '')).strip()

def _read(source):
    dlg = read_ipynb(source)
    if not dlg: raise ValueError(f'could not read dialog {source}')
    return dlg

def _prompts(dlg): return [m for m in dlg if m.msg_type == sprompt and not m.skipped]

def _text(msg): return ''.join(p.text for p in msg.content if isinstance(p, Text) and p.text)

## Capture a session

Capture reads an archive after the conversation ended; it never watches a live process. Ramabana
goes through `read_history`, scoring on the way past. Claude and Codex go through llmsurgery, whose
reader resolves an id without being told its host — so Drona checks the host it got back.

In [ ]:
#| export
LS_HOSTS = {'claude': 'ant', 'codex': 'oai'}

def turn_msgs(
    turn, # one Ramabana archive turn
    n=0,  # the turn's place in the round, which keeps fallback call ids unique
):
    "One Ramabana archive turn as typed Aidialog messages."
    msgs = [Msg('user', [Text(str(turn.get('prompt') or ''))])]
    for i, a in enumerate(turn.get('activity') or ()):
        cid, tool = a.get('action_id') or a.get('id') or f'call_{n}_{i}', a.get('tool', '')
        args, detail = a.get('args') or {}, str(a.get('detail') or '')
        msgs += [Msg('assistant', [ToolUse(id=cid, name=tool, arguments=args)]),
                 Msg('tool', [ToolResult(id=cid, name=tool, arguments=args, text=detail)])]
    if reply := str(turn.get('reply') or ''): msgs.append(Msg('assistant', [Text(reply)]))
    return msgs

def _latest_path(host, cwd, codex_home):
    "The newest session file on `host`."
    if host == 'codex': return oai.project_thread(cwd or '.', codex_home or oai.CODEX_HOME)[1]
    paths = sorted(ant.sess_dir(cwd).glob('*.jsonl'), key=lambda p: p.stat().st_mtime)
    if not paths: raise ValueError(f'no Claude session under {ant.sess_dir(cwd)}')
    return paths[-1]

def host_dlg(
    host,             # `claude` or `codex`
    session='latest', # session id, id prefix, or `latest`
    cwd=None,         # project directory
    codex_home=None,  # Codex home
    name=None,        # dialog name
):
    "One Claude or Codex session as a dialog, checked against the host that was asked for."
    if session == 'latest':
        path = _latest_path(host, cwd, codex_home)
        dlg = path_dlg(LS_HOSTS[host], path, name=name or path.stem, mx=None)
    else: dlg = sess_dlg(session, cwd=cwd, codex_home=codex_home, name=name, mx=None)
    got = (dlg.meta.get('llmsurgery') or {}).get('host')
    if got != LS_HOSTS[host]: raise ValueError(f'session {session!r} belongs to {got!r}, not {host!r}')
    return dlg

def capture(
    output,                   # review notebook path
    host='ramabana',          # `ramabana`, `claude`, or `codex`
    session='latest',         # session id, id prefix, or `latest`
    cwd=None,                 # project directory, for Claude and Codex
    history=RAMABANA_HISTORY, # Ramabana history path
    codex_home=None,          # Codex home
    name=None,                # round name; the output stem when omitted
):
    "Capture one host session as a Drona review notebook."
    if host not in HOSTS: raise ValueError(f'host must be one of {HOSTS}')
    output, name, session = Path(output), name or Path(output).stem, session or 'latest'
    if host == 'ramabana':
        turns = read_history(history, session)
        dlg = chat2dlg([m for n, t in enumerate(turns) for m in turn_msgs(t, n)], name, mx=None)
        found = {'session': turns[0].get('session'), **assess_history(turns).dict()}
    else:
        dlg = host_dlg(host, session, cwd, codex_home, name)
        found = dict(dlg.meta.get('llmsurgery') or {})
    review = {**found, 'version': drona_version(), 'status': 'review', 'host': host}
    dlg.meta[REVIEW_KEY] = review
    dlg.mk_message(REVIEW_NOTE, idx=0, msg_type=snote, skipped=1, meta={REVIEW_KEY: review})
    output.parent.mkdir(parents=True, exist_ok=True)
    write_ipynb(dlg, output)
    return output

The score rides in the metadata, so a reviewer knows where to look before reading.

In [ ]:
tmp = Path(tempfile.mkdtemp())
archive = tmp/'agent-history.jsonl'
turn = {'session': 'sess-aaa', 'state': 'complete',
        'prompt': 'Use fossick to research the AnswerDotAI llmdojo github repository',
        'reply': 'FOSSICK read the repository directly.',
        'activity': [{'action_id': 'a0', 'tool': 'run_shell', 'ok': True,
                      'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'},
                      'detail': '# llmdojo\nLLM coding agents imitate what their context shows.'}]}
archive.write_text(json.dumps(turn) + '\n')

review = capture(tmp/'round.ipynb', history=archive)
captured = read_ipynb(review)
test_eq(captured.meta[REVIEW_KEY]['status'], 'review')
test_eq(captured.meta[REVIEW_KEY]['host'], 'ramabana')
test_eq(captured.meta[REVIEW_KEY]['score'], 100)
test_fail(lambda: capture(tmp/'x.ipynb', host='other'), contains='host must be one of')

Fallback call ids count from the start of the round, not of each turn, so a multi-turn round whose records predate `action_id` still accepts.

In [ ]:
bare = dict(turn, activity=[{k: v for k, v in turn['activity'][0].items() if k != 'action_id'}])
(tmp/'multi.jsonl').write_text('\n'.join(json.dumps(dict(bare, prompt=p)) for p in ('one', 'two')) + '\n')
ids = [p.id for n, t in enumerate(read_history(tmp/'multi.jsonl'))
       for m in turn_msgs(t, n) for p in m.content if isinstance(p, ToolUse)]
test_eq(len(set(ids)), 2)

## Accept a round

Explicit and attributed. It checks the reviewed dialog still round trips to a history opening with a
user turn, records who accepted it, and writes that history beside the notebook.

In [ ]:
#| export
def _part_dict(part):
    kind = getattr(part.type, 'value', None) or str(part.type)
    return {'type': kind, **{k: v for k, v in vars(part).items()
                             if k != 'raw' and v not in (None, False, {}, [])}}

def _msg_dict(msg): return {'role': msg.role, 'content': [_part_dict(p) for p in msg.content]}

def _reviewed(dlg, source):
    "The prompt turns a reviewer left in place, refusing a round they emptied."
    prompts = _prompts(dlg)
    if not prompts: raise ValueError(f'{source} has no reviewed prompt turns left')
    return Dialog(prompts, name=dlg.name, meta=dlg.meta)

def accepted(source):
    "The reviewed prompts of an accepted round, refusing a round nobody has accepted."
    dlg = _read(source)
    if (dlg.meta.get(REVIEW_KEY) or {}).get('status') != 'accepted':
        raise ValueError(f'{source} is not accepted; run drona-accept on it first')
    return _reviewed(dlg, source)

def compiled_history(source):
    "Canonical Aidialog history from an accepted round."
    return dlg2chat(accepted(source), plain=True)

def accept(
    source,      # reviewed Drona notebook
    reviewer,    # the person accepting it
    output=None, # compiled JSON path; `<source>.json` when omitted
):
    "Accept a reviewed round and write its canonical history."
    if not str(reviewer).strip(): raise ValueError('a round needs a named reviewer')
    dlg = _read(source)
    history = dlg2chat(_reviewed(dlg, source), plain=True)
    if not history or history[0].role != 'user': raise ValueError('a round must open with a user turn')
    if PENDING in _text(history[-1]):
        raise ValueError('the last turn of this round was never answered; answer or skip it first')
    meta = {**(dlg.meta.get(REVIEW_KEY) or {}), 'status': 'accepted',
            'reviewer': str(reviewer), 'accepted_version': drona_version()}
    dlg.meta[REVIEW_KEY] = meta
    dlg.save(source)
    output = Path(output) if output else Path(source).with_suffix('.json')
    output.write_text(json.dumps({'meta': meta, 'history': [_msg_dict(m) for m in history]}, indent=2))
    return output

In [ ]:
test_fail(lambda: compiled_history(review), contains='is not accepted')
test_fail(lambda: accept(review, '  '), contains='named reviewer')

compiled = json.loads(accept(review, 'Karthik').read_text())
test_eq(compiled['meta']['status'], 'accepted')
test_eq(compiled['meta']['reviewer'], 'Karthik')
test_eq(compiled['history'][0]['role'], 'user')
test_eq([m.role for m in compiled_history(review)], ['user', 'assistant', 'tool', 'assistant'])

A round a reviewer emptied, and one whose last turn nobody answered, are both refused: neither is a route to imitate.

In [ ]:
from aidialog.ipynb import write_ipynb

emptied = capture(tmp/'emptied.ipynb', history=archive)
dlg = read_ipynb(emptied)
for m in dlg:
    if m.msg_type == sprompt: m.skipped = 1
write_ipynb(dlg, emptied)
test_fail(lambda: accept(emptied, 'Karthik'), contains='no reviewed prompt turns left')

unanswered = capture(tmp/'unanswered.ipynb', history=archive)
dlg = read_ipynb(unanswered)
dlg.mk_message('and what about the tests?', msg_type=sprompt)
write_ipynb(dlg, unanswered)
test_fail(lambda: accept(unanswered, 'Karthik'), contains='never answered')

accept(capture(tmp/'multi.ipynb', history=tmp/'multi.jsonl'), 'Karthik')

## Compile a round

Three shapes come out: Urai history for a chat, a bootstrap prompt for a host whose command line
takes no prepared history, or a host session file.

In [ ]:
#| export
def warm_start(source):
    "An accepted round as Urai history, for `messages=` on any Urai or Rishi chat."
    out = []
    for m in compiled_history(source):
        if m.role == 'user':
            out.append({'role': 'user', 'content': _plain(m.text)})
        elif m.role == 'assistant':
            calls = [ToolCall(p.name, p.arguments, id=p.id) for p in m.content if isinstance(p, ToolUse)]
            out.append({'role': 'assistant', 'content': _plain(_text(m)),
                        **({'tool_calls': calls} if calls else {})})
        elif m.role == 'tool':
            out += [mk_tool_res_msg(ToolCall(p.name, p.arguments, id=p.id), p.text)
                    for p in m.content if isinstance(p, ToolResult)]
    return out

def prepare_chat(
    chat,   # an empty Urai or Rishi chat
    source, # accepted round notebook
):
    "Prepend an accepted round to an empty Urai-compatible chat."
    if chat.hist: raise ValueError('Drona prepares an empty chat only')
    chat.hist = chat.fmt2hist(warm_start(source))
    if hasattr(chat, '_recreate_conv'): chat._recreate_conv()
    return chat

def bootstrap_prompt(
    source,                  # accepted round notebook
    detail=BOOTSTRAP_DETAIL, # characters of each tool result to keep
):
    "An accepted round as one prompt, for a host whose command line takes no prepared history."
    rows = ['The reviewed Drona round below is the tool route to follow.']
    for m in compiled_history(source):
        if m.role == 'user':
            rows.append(f'User: {_plain(m.text)}')
            continue
        for p in m.content:
            if isinstance(p, ToolUse):
                rows.append(f'Assistant tool: {p.name}({json.dumps(p.arguments, sort_keys=True, default=str)})')
            elif isinstance(p, ToolResult): rows.append(f'Tool result: {_clip(p.text, detail)}')
            elif isinstance(p, Text) and _plain(p.text): rows.append(f'Assistant: {_plain(p.text)}')
    rows.append('Reply with exactly: DRONA_READY')
    return '\n\n'.join(rows)

def export_round(
    source,      # accepted round notebook
    host,        # `ramabana`, `claude`, or `codex`
    output=None, # file to write; Claude writes into its own session directory instead
    cwd=None,    # project directory, for Claude
):
    "Compile an accepted round for one host."
    if host not in HOSTS: raise ValueError(f'host must be one of {HOSTS}')
    if host == 'claude': return ant.dlg2sess(accepted(source), cwd=cwd)
    out = bootstrap_prompt(source) if host == 'ramabana' else list(oai.dlg2items(accepted(source)))
    if output: Path(output).write_text(out if isinstance(out, str) else json.dumps(out, indent=2))
    return out

Urai history keeps the tool call and its result paired by the same id.

In [ ]:
hist = warm_start(review)
test_eq([m['role'] for m in hist], ['user', 'assistant', 'tool', 'assistant'])
test_eq(hist[1]['tool_calls'][0].name, 'run_shell')
test_eq(hist[2]['tool_call_id'], hist[1]['tool_calls'][0]['id'])

An attachment does not travel with a round, so the markers that refer to one are dropped too. A later session is never told about an image it was not given.

In [ ]:
from aidialog.msg_parts import InputImage

pixel = ('data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8'
         'z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg==')
shot = chat2dlg([Msg('user', [Text('look at this'), InputImage(text=pixel, mime='image/png')]),
                 Msg('assistant', [Text('I see it.')])], 'shot', mx=None)
shot.meta[REVIEW_KEY] = {'status': 'review'}
write_ipynb(shot, tmp/'shot.ipynb')
accept(tmp/'shot.ipynb', 'Karthik')

for text in (warm_start(tmp/'shot.ipynb')[0]['content'], bootstrap_prompt(tmp/'shot.ipynb')):
    assert '<media' not in text and 'attachment:' not in text
test_eq(warm_start(tmp/'shot.ipynb')[0]['content'], 'look at this')

`prepare_chat` puts that history on an empty chat and refuses one that already has some. A chat need only carry `hist` and `fmt2hist`, so no model is loaded.

In [ ]:
class FakeChat:
    def __init__(self): self.hist = []
    def fmt2hist(self, msgs): return list(msgs)

chat = prepare_chat(FakeChat(), review)
test_eq([m['role'] for m in chat.hist], ['user', 'assistant', 'tool', 'assistant'])
test_fail(lambda: prepare_chat(chat, review), contains='empty chat only')

One pass over the whole compile stage: every host refuses an unreviewed notebook, a long tool result is clipped rather than pasted onto a command line, and Codex keeps one call id across the function call and its output.

In [ ]:
big = dict(turn, activity=[dict(turn['activity'][0], detail='x'*5000)])
(tmp/'big.jsonl').write_text(json.dumps(big) + '\n')
accept(capture(tmp/'big.ipynb', history=tmp/'big.jsonl'), 'Karthik')
assert len(bootstrap_prompt(tmp/'big.ipynb')) < 2000
assert 'more chars]' in bootstrap_prompt(tmp/'big.ipynb')

## Start the next session

Ramabana takes no prepared history on its command line, so `start_round` sends the round as one
bootstrap turn and resumes the session it creates. Ramabana exits non-zero whenever a turn had
anything to report, so the bootstrap's code is surfaced and the resume goes ahead regardless.

In [ ]:
#| export
def start_commands(
    source,     # accepted round notebook
    root='.',   # Ramabana root folders, comma separated
    model=None, # optional model name
    cfg=None,   # Ramabana config dir, when it is not the default
):
    "The Ramabana bootstrap and resume commands for an accepted round."
    base = ['ramabana', '--root', str(root)]
    if cfg: base += ['--cfg', str(cfg)]
    if model: base += ['--model', model]
    return base + [bootstrap_prompt(source)], base + ['--resume', 'latest']

def start_round(
    source,      # accepted round notebook
    root='.',    # Ramabana root folders, comma separated
    model=None,  # optional model name
    cfg=None,    # Ramabana config dir, when it is not the default
    launch=False, # run the commands, rather than return them
):
    "Prepare or launch Ramabana with an accepted Drona round."
    first, resume = start_commands(source, root, model, cfg)
    if not launch: return {'bootstrap': first, 'resume': resume}
    if rc := subprocess.run(first).returncode: print(f'bootstrap exited {rc}', file=sys.stderr)
    return subprocess.run(resume).returncode

In [ ]:
first, resume = start_commands(review, root='/tmp/project', model='sonnet', cfg='/tmp/cfg')
test_eq(first[:8], ['ramabana', '--root', '/tmp/project', '--cfg', '/tmp/cfg', '--model', 'sonnet',
                    bootstrap_prompt(review)])
test_eq(resume[-2:], ['--resume', 'latest'])
test_eq(start_round(review, root='/tmp/project')['resume'][:3], ['ramabana', '--root', '/tmp/project'])

## Command line

Five commands follow a round's life: assess, capture, accept, compile, start.

In [ ]:
#| export
@call_parse
def capture_cli(output: str, host: str='ramabana', session: str='latest', cwd: str=None,
                history: str=str(RAMABANA_HISTORY), codex_home: str=None, name: str=None):
    "Capture a host session as a Drona review notebook."
    print(report(capture, output, host, session, cwd, history, codex_home, name))

@call_parse
def accept_cli(source: str, reviewer: str, output: str=None):
    "Accept a reviewed round and compile its history."
    print(report(accept, source, reviewer, output))

@call_parse
def export_cli(source: str, host: str, output: str=None, cwd: str=None):
    "Compile an accepted round for one host."
    out = report(export_round, source, host, output, cwd)
    if output and host != 'claude': print(output)
    else: print(out if isinstance(out, str) else json.dumps(out, indent=2))

@call_parse
def start_cli(source: str, root: str='.', model: str=None, cfg: str=None, launch: bool=False):
    "Prepare or launch Ramabana with an accepted round."
    out = report(start_round, source, root, model, cfg, launch)
    if not isinstance(out, dict): sys.exit(out)
    for name, cmd in out.items(): print(f'{name}: {shlex.join(cmd)}')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()